In [ ]:
# ============================================================
# CELL: Sanity check install (UPDATED)
# Changed: removed `import paddle` immediately followed by paddleocr use in
# the same flow — now checks torch AND paddle, and deliberately does NOT
# import paddleocr here. Importing paddleocr initializes PaddleX's global
# registry; doing that in a "just checking" cell risks a partial failure
# that then blocks the real import later with a RuntimeError.
# ============================================================
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

import paddle
print("paddle:", paddle.__version__, "| CUDA:", paddle.device.is_compiled_with_cuda())

In [ ]:
# ============================================================
# CELL: Git-lfs pull sample data — UNCHANGED from original notebook
# ============================================================
!apt-get -qq install git-lfs
!git lfs install

!git clone --depth 1 https://github.com/bankh/CatalogBank.git /content/CatalogBank
%cd /content/CatalogBank
!git lfs pull -I "Catalogs/Sample/Thorlabs/OptoMechanics_v21/_pdfs/*,Catalogs/Sample/Thorlabs/OptoMechanics_v21/_images/*,Catalogs/Sample/McMasterCarr/_pdfs/*,Catalogs/Sample/McMasterCarr/_images/*"
%cd /content

In [ ]:
# ============================================================
# CELL: Pick sample pages — UNCHANGED
# ============================================================
import glob, random, os

thorlabs_pdfs = sorted(glob.glob("/content/CatalogBank/Catalogs/Sample/Thorlabs/OptoMechanics_v21/_pdfs/*.pdf"))
mcmaster_pdfs = sorted(glob.glob("/content/CatalogBank/Catalogs/Sample/McMasterCarr/_pdfs/*.pdf"))

random.seed(42)
sample_pdfs = random.sample(thorlabs_pdfs, min(3, len(thorlabs_pdfs))) + \
              random.sample(mcmaster_pdfs, min(3, len(mcmaster_pdfs)))
print(f"{len(sample_pdfs)} sample pages selected")

In [ ]:
# ============================================================
# CELL: Rasterize PDFs to images — UNCHANGED
# ============================================================
import fitz  # pymupdf

os.makedirs("/content/page_images", exist_ok=True)

def pdf_to_image(pdf_path, out_dir="/content/page_images", dpi=200):
    doc = fitz.open(pdf_path)
    page = doc[0]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(pdf_path))[0] + ".png")
    pix.save(out_path)
    doc.close()
    return out_path

image_paths = [pdf_to_image(p) for p in sample_pdfs]
image_paths

In [ ]:
# ============================================================
# CELL: Run structure extraction (REPLACED)
# The old cell used `from paddleocr import PPStructure` and called it as
# `structure_engine(img_path)`, returning a list of dicts with `type`,
# `bbox`, `res`. That class is 2.x-era API and no longer matches what
# paddleocr 3.7.0 installs. Replaced with PPStructureV3, using its
# `.predict()` + `.save_to_json()` / `.save_to_markdown()` interface.
# This must be the FIRST paddleocr import this session — run it once.
# ============================================================
from paddleocr import PPStructureV3

os.makedirs("/content/output/ppstructurev3", exist_ok=True)

pp_structure = PPStructureV3()

pp_structure_results = {}
for img_path in image_paths:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    out_dir = f"/content/output/ppstructurev3/{stem}"
    os.makedirs(out_dir, exist_ok=True)

    for res in pp_structure.predict(img_path):
        res.save_to_json(save_path=out_dir)
        res.save_to_markdown(save_path=out_dir)
        res.save_to_img(save_path=out_dir)   # built-in visualization, replaces the old cv2 draw_regions() cell
        pp_structure_results[img_path] = res
    print(f"{stem}: done -> {out_dir}")

In [ ]:
# ============================================================
# CELL: Inspect actual output schema before normalizing (NEW)
# I don't want to hand you a normalize_page() function that guesses at
# JSON key names for the 3.x output — the schema differs from the old
# `type`/`bbox`/`res` dict shape. Run this first and look at the printed
# keys/structure, then we write the real normalizer against what's
# actually there.
# ============================================================
import json

sample_stem = os.path.splitext(os.path.basename(image_paths[0]))[0]
json_path = glob.glob(f"/content/output/ppstructurev3/{sample_stem}/*.json")[0]

with open(json_path) as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print(json.dumps(data, indent=2)[:2000])

In [ ]:
# ============================================================
# CELL: Visualize — REPLACED
# The old cell manually drew bboxes with cv2 based on region['bbox']/['type'].
# That schema doesn't apply anymore. res.save_to_img() above already wrote
# an annotated visualization to disk — just display it.
# ============================================================
from PIL import Image
import matplotlib.pyplot as plt

vis_path = glob.glob(f"/content/output/ppstructurev3/{sample_stem}/*.png")[0]
img = Image.open(vis_path)
plt.figure(figsize=(10,14))
plt.imshow(img)
plt.axis("off")
plt.show()